# Model Prediction on Train, Validation and Test Sets

In [15]:
import pandas as pd
from sklearn.svm import SVC
import numpy as np
from sklearn.metrics import classification_report
from joblib import load

In [6]:
data_dir = 'data/final_data/'
train_df = pd.read_csv(data_dir + 'train_data_with_features.csv')
val_df = pd.read_csv(data_dir + 'val_data_with_features.csv')
test_df = pd.read_csv(data_dir + 'test_data_with_features.csv')

In [16]:
train_df.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f3467,f3468,f3469,f3470,f3471,f3472,label,image_path,cell_number,cell_image_path
0,0.002533,0.025067,0.007200,0.087733,0.114267,0.360533,0.070933,0.053333,0.233733,0.044667,...,91.80600,12.529564,-0.218953,201.73093,9.080140,-1.050871,no_object,data/final_data/resized_images/299784.webp,58,data/final_data/grid_cells/299784/c58.jpg
1,0.003333,0.022933,0.015200,0.123200,0.386933,0.302400,0.054000,0.026400,0.041467,0.024133,...,68.47880,43.146996,1.002107,85.10947,30.384699,0.210047,no_object,data/final_data/resized_images/648ff941aa8f.webp,42,data/final_data/grid_cells/648ff941aa8f/c42.jpg
2,0.000933,0.011333,0.008133,0.106133,0.301200,0.320933,0.041200,0.033733,0.162267,0.014133,...,136.54227,71.812650,0.147966,197.52693,6.737097,-0.371850,no_object,data/final_data/resized_images/409575.webp,39,data/final_data/grid_cells/409575/c39.jpg
3,0.001867,0.016533,0.004933,0.082667,0.303600,0.398533,0.033467,0.021600,0.108133,0.028667,...,57.38000,12.993029,-1.470382,173.22853,61.692570,-0.751049,no_object,data/final_data/resized_images/408828.jpeg,55,data/final_data/grid_cells/408828/c55.jpg
4,0.000533,0.007600,0.001067,0.081067,0.014133,0.282400,0.034933,0.058800,0.508400,0.011067,...,193.43120,4.218049,0.029969,106.22773,1.987395,-0.120476,no_object,data/final_data/resized_images/306975.webp,4,data/final_data/grid_cells/306975/c04.jpg


In [8]:
# Combine train and validation sets as training data
combined_train_df = pd.concat([train_df, val_df], ignore_index=True)

In [17]:
combined_train_df.head()

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f3467,f3468,f3469,f3470,f3471,f3472,label,image_path,cell_number,cell_image_path
0,0.002533,0.025067,0.007200,0.087733,0.114267,0.360533,0.070933,0.053333,0.233733,0.044667,...,91.80600,12.529564,-0.218953,201.73093,9.080140,-1.050871,no_object,data/final_data/resized_images/299784.webp,58,data/final_data/grid_cells/299784/c58.jpg
1,0.003333,0.022933,0.015200,0.123200,0.386933,0.302400,0.054000,0.026400,0.041467,0.024133,...,68.47880,43.146996,1.002107,85.10947,30.384699,0.210047,no_object,data/final_data/resized_images/648ff941aa8f.webp,42,data/final_data/grid_cells/648ff941aa8f/c42.jpg
2,0.000933,0.011333,0.008133,0.106133,0.301200,0.320933,0.041200,0.033733,0.162267,0.014133,...,136.54227,71.812650,0.147966,197.52693,6.737097,-0.371850,no_object,data/final_data/resized_images/409575.webp,39,data/final_data/grid_cells/409575/c39.jpg
3,0.001867,0.016533,0.004933,0.082667,0.303600,0.398533,0.033467,0.021600,0.108133,0.028667,...,57.38000,12.993029,-1.470382,173.22853,61.692570,-0.751049,no_object,data/final_data/resized_images/408828.jpeg,55,data/final_data/grid_cells/408828/c55.jpg
4,0.000533,0.007600,0.001067,0.081067,0.014133,0.282400,0.034933,0.058800,0.508400,0.011067,...,193.43120,4.218049,0.029969,106.22773,1.987395,-0.120476,no_object,data/final_data/resized_images/306975.webp,4,data/final_data/grid_cells/306975/c04.jpg


In [18]:
# Load model
final_model_path = "final_model/SVM_1213_232905.pkl"
with open(final_model_path, "rb") as f:
    clf = load(f)
clf

,C,np.float64(100.0)
,kernel,'rbf'
,degree,3
,gamma,'auto'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [19]:
# Load prescaler, postscaler and PCA
with open("final_model/pre_scaler.pkl", "rb") as f:
    pre_scaler = load(f)
with open("final_model/post_scaler.pkl", "rb") as f:
    post_scaler = load(f)
with open("final_model/pca.pkl", "rb") as f:
    pca = load(f)

In [29]:
label_mapping = {'no_object': 0, 'ball': 1, 'bat': 2, 'stump': 3}

In [30]:
def predict(model, pre_scaler, post_scaler, pca, data, data_type='test'):
    X = data.drop(columns=['image_path', 'label', 'cell_number', 'cell_image_path'])

    X_scaled = pre_scaler.transform(X)
    X_pca = pca.transform(X_scaled)
    X_post_scaled = post_scaler.transform(X_pca)

    y_pred = model.predict(X_post_scaled)

    # Create new dataframe with original columns and predictions
    df = data[['image_path', 'cell_number', 'cell_image_path']].copy()
    df['predicted_label'] = y_pred
    df['data_type'] = data_type

    df['true_label'] = data['label'].map(label_mapping).values


    return df


In [31]:
output = predict(clf, pre_scaler, post_scaler, pca, combined_train_df, data_type='train')

In [37]:
output.columns

Index(['image_path', 'cell_number', 'cell_image_path', 'predicted_label',
       'data_type', 'true_label'],
      dtype='object')

In [32]:
output.predicted_label.value_counts()

predicted_label
0    12004
2     3348
3     1736
1      320
Name: count, dtype: int64

In [35]:
print(classification_report(output['true_label'], output['predicted_label']))

              precision    recall  f1-score   support

           0       0.99      0.77      0.87     15580
           1       0.73      0.84      0.78       281
           2       0.22      0.93      0.36       796
           3       0.42      0.96      0.58       751

    accuracy                           0.78     17408
   macro avg       0.59      0.87      0.65     17408
weighted avg       0.93      0.78      0.83     17408



In [41]:
def format_df_for_saving(df, data_type='test'):
    # Aggregate all the cell predictions for each image and store them in columns c1, c2, c3, ...
    formatted_data = []
    for image_path, group in df.groupby('image_path'):
        row = {'ImageFileName': image_path, 'TrainOrTest': data_type}
        
        cell_label_vector = [-1] * 64
        for _, cell in group.iterrows():
            cell_label_vector[cell["cell_number"] - 1] = cell['predicted_label']
        
        assert not any(label == -1 for label in cell_label_vector), f"Missing cell predictions for image {image_path}"

        for i in range(64):
            cell_col = f'c{i+1}'
            row[cell_col] = cell_label_vector[i]
        formatted_data.append(row)
    return pd.DataFrame(formatted_data)
    

In [42]:
output_train_df = format_df_for_saving(output, data_type='train')

In [43]:
output_train_df.head()

,ImageFileName,TrainOrTest,c1,c2,c3,c4,c5,c6,c7,c8,...,c55,c56,c57,c58,c59,c60,c61,c62,c63,c64
0,data/final_data/resized_images/0b70cbb67c0c.webp,train,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,data/final_data/resized_images/107384394.jpg,train,2,0,0,0,1,0,0,0,...,3,0,3,3,0,0,0,0,3,0
2,data/final_data/resized_images/1132922938ee.webp,train,2,0,0,0,0,0,0,0,...,0,0,3,0,0,0,0,0,0,0
3,data/final_data/resized_images/113af926a798.webp,train,0,0,0,0,0,3,0,3,...,0,0,0,0,0,0,2,0,0,0
4,data/final_data/resized_images/1698499317-6258...,train,0,0,0,0,0,0,0,0,...,2,2,3,0,0,0,0,0,0,2


In [44]:
test_output = predict(clf, pre_scaler, post_scaler, pca, test_df, data_type='test')

In [45]:
test_output.predicted_label.value_counts()

predicted_label
0    2281
2     479
3     278
1      34
Name: count, dtype: int64

In [46]:
print(classification_report(test_output['true_label'], test_output['predicted_label']))

              precision    recall  f1-score   support

           0       0.96      0.80      0.87      2750
           1       0.38      0.23      0.29        57
           2       0.13      0.56      0.21       109
           3       0.41      0.74      0.53       156

    accuracy                           0.78      3072
   macro avg       0.47      0.58      0.47      3072
weighted avg       0.89      0.78      0.82      3072



In [47]:
output_test_df = format_df_for_saving(test_output, data_type='test')

In [48]:
output_test_df.head()

,ImageFileName,TrainOrTest,c1,c2,c3,c4,c5,c6,c7,c8,...,c55,c56,c57,c58,c59,c60,c61,c62,c63,c64
0,data/final_data/resized_images/274292.webp,test,0,0,0,0,0,0,0,0,...,3,0,0,0,0,0,0,0,3,0
1,data/final_data/resized_images/298739.webp,test,0,0,0,0,0,2,0,0,...,0,0,0,0,3,0,0,0,0,0
2,data/final_data/resized_images/306307.webp,test,0,0,2,0,0,3,0,0,...,0,0,0,0,0,0,3,0,0,0
3,data/final_data/resized_images/306309.webp,test,0,0,0,0,0,2,0,0,...,0,0,0,3,0,0,0,0,0,2
4,data/final_data/resized_images/306962.webp,test,0,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [49]:
# Merge train and test output dataframes
final_output_df = pd.concat([output_train_df, output_test_df], ignore_index=True)

In [50]:
import os
os.makedirs('output', exist_ok=True)
final_output_df.to_csv('output/final_predictions.csv', index=False)

In [55]:
# For all the images in final_output_df, create output images with a grid overlay showing each cell coloured by predicted label
# Colour map for labels - no_object: none, ball: green, bat: red, stump: blue
# Save these images in output/image_predictions_all/ directory

import cv2
import os
from PIL import Image, ImageDraw

def create_prediction_images(input_df, output_dir):
    """
    Create output images with grid overlay showing predicted labels for each cell.
    
    Parameters:
    -----------
    input_df : pd.DataFrame
        DataFrame containing ImageFileName and prediction columns (c1-c64)
    output_dir : str
        Directory path where output images will be saved
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Color map for labels (BGR format for OpenCV)
    color_map = {
        0: None,  # no_object: no color (transparent)
        1: (0, 255, 0),  # ball: green
        2: (0, 0, 255),  # bat: red
        3: (255, 0, 0)   # stump: blue
    }
    
    # Process each image
    for idx, row in input_df.iterrows():
        image_path = row['ImageFileName']
        
        # Load the original image
        img = cv2.imread(image_path)
        if img is None:
            print(f"Could not load image: {image_path}")
            continue
        
        height, width = img.shape[:2]
        cell_height = height // 8
        cell_width = width // 8
        
        # Create an overlay for semi-transparent coloring
        overlay = img.copy()
        
        # Draw grid and color cells based on predictions
        for cell_idx in range(64):
            cell_col = f'c{cell_idx + 1}'
            predicted_label = int(row[cell_col])
            
            # Calculate cell position
            row_idx = cell_idx // 8
            col_idx = cell_idx % 8
            
            x1 = col_idx * cell_width
            y1 = row_idx * cell_height
            x2 = x1 + cell_width
            y2 = y1 + cell_height
            
            # Fill cell with color if not no_object
            color = color_map.get(predicted_label)
            if color is not None:
                cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
        
        # Blend the overlay with the original image (30% opacity for colors)
        alpha = 0.4
        img_with_overlay = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
        
        # Draw grid lines
        for i in range(1, 8):
            # Vertical lines
            cv2.line(img_with_overlay, (i * cell_width, 0), (i * cell_width, height), (255, 255, 255), 2)
            # Horizontal lines
            cv2.line(img_with_overlay, (0, i * cell_height), (width, i * cell_height), (255, 255, 255), 2)
        
        # Save the output image
        output_filename = os.path.basename(image_path)
        output_path = os.path.join(output_dir, output_filename)
        cv2.imwrite(output_path, img_with_overlay)
        
        if (idx + 1) % 10 == 0:
            print(f"Processed {idx + 1}/{len(input_df)} images")
    
    print(f"All images saved to {output_dir}")


In [56]:
# Generate prediction images for test data
create_prediction_images(output_test_df, 'output/image_predictions_test/')

Processed 10/48 images
Processed 20/48 images
Processed 20/48 images
Processed 30/48 images
Processed 30/48 images
Processed 40/48 images
Processed 40/48 images
All images saved to output/image_predictions_test/
All images saved to output/image_predictions_test/


In [54]:
# Generate prediction images for all data
create_prediction_images(output_train_df, 'output/image_predictions_train/')

Processed 10/272 images
Processed 20/272 images
Processed 20/272 images
Processed 30/272 images
Processed 30/272 images
Processed 40/272 images
Processed 40/272 images
Processed 50/272 images
Processed 50/272 images
Processed 60/272 images
Processed 60/272 images
Processed 70/272 images
Processed 70/272 images
Processed 80/272 images
Processed 80/272 images
Processed 90/272 images
Processed 100/272 images
Processed 90/272 images
Processed 100/272 images
Processed 110/272 images
Processed 110/272 images
Processed 120/272 images
Processed 120/272 images
Processed 130/272 images
Processed 130/272 images
Processed 140/272 images
Processed 140/272 images
Processed 150/272 images
Processed 150/272 images
Processed 160/272 images
Processed 160/272 images
Processed 170/272 images
Processed 170/272 images
Processed 180/272 images
Processed 180/272 images
Processed 190/272 images
Processed 190/272 images
Processed 200/272 images
Processed 200/272 images
Processed 210/272 images
Processed 210/272